# Exercise: Human activity dataset

We treat here the human activity dataset that can be found [here](https://archive.ics.uci.edu/dataset/240/human+activity+recognition+using+smartphones) or [here.](https://www.openml.org/search?type=data&status=active&id=1478)

In the dataset, each datapoint represents a human performing different activities, eg, walking, standing, sitting etc. The data consists of features computed mainly from acceleration data recorded by smartphones. See the links for more details. The objective lies in recognising the activity from the movement data. 

In this exercise we will compare a neural network trained with a constant learning rate to one trained with cosine annealing.

First we download the dataset, which conviently is available on OpenML.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import fetch_openml

# Fetch the Human Activity Recognition dataset by its OpenML ID
har_data = fetch_openml(data_id=1478, as_frame=True, parser='auto')

# Extract features (X) and target labels (y)
X = har_data.data
y = har_data.target

# View the structure of the feature matrix
display(X.head())

The classes are stored as strings. 

In [ ]:
set(y)

We use <code>LabelEncoder</code> from scikit-learn to turn the classes into integer classes. We also set up a method to split the data randomly into a training and a test set.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Label Encoding
# TensorFlow requires numeric targets for the loss function
encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)

# Print the exact integer-to-class mapping for reference
class_mapping = dict(enumerate(encoder.classes_))
print(f"Class mapping: {class_mapping}")

# Train/Test Split
def get_train_test_split():
    X_train, X_test, y_train, y_test = train_test_split(
        X.astype(np.float32), # Ensure float32 for TF weights
        y_encoded, 
        test_size=0.3, 
    )
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = get_train_test_split()

# Set up simple neural network

We set up a simple neural network with two hidden layers. We also include [dropout](https://en.wikipedia.org/wiki/Dropout_(neural_networks)) to prevent overfitting. For the loss function we use cross entropy, which is standard. For the optimiser we choose AdamW, a variant of stochastic gradient descent.

We fix a constant learning rate. The value here was obtained by hyperparameter tuning that is not shown in the exercise notebook. The code will be included in the solution notebook.

In [ ]:
import tensorflow as tf

# Define sequential, fully connected model
# dropout layers against overfitting (really necessary?)
def setup_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(X_train.shape[1],)),
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(6, activation='softmax')
    ])
    return model   

model = setup_model()

# Compilation
# SparseCategoricalCrossentropy computes the cross-entropy directly from integer labels,
# avoiding the memory overhead of constructing one-hot encoded matrices.

def model_compile(model, learning_rate):
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy']
    )

# found by hyperparameter tuning
best_constant_lr = 0.006
model_compile(model,best_constant_lr)

We fix training a bit arbitrarily to 30 epochs and batch size to 64. We also specify that 20% of the training data will be used for validation.

In [ ]:
# Training
epochs = 30
batch_size = 64
history = model.fit(
    X_train, 
    y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_split=0.2, # Hold out 20% of the training data for validation
    verbose="auto"
)

# Evaluation on the test set
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"\nTest Accuracy: {test_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")

The history object contains different metrics recorded during training. Let's plot training loss.

In [ ]:
import matplotlib.pyplot as plt

fig,axs=plt.subplots(1,2,figsize=(10,3))
axs[0].plot(history.history["loss"],label="train",alpha=0.7)
axs[0].plot(history.history["val_loss"],label="validation",alpha=0.7)
axs[0].set_xlabel("epochs")
axs[0].set_ylabel("training loss")
axs[0].legend()

axs[1].plot(history.history["accuracy"],label="train",alpha=0.7)
axs[1].plot(history.history["val_accuracy"],label="validation",alpha=0.7)
axs[1].set_xlabel("epochs")
axs[1].set_ylabel("accuracy")
axs[1].legend()

# ...and with cosine annealing 

Next, we repeat the training but this time use cosine annealing for the learning rate. 

## Task:
* Find out how to do cosine annealing with tensorflow. If you do not know how, ask an AI.
* Cosine annealing will need to parameters, an initial learning rate <code>initial_learning_rate</code> and a parameter that basically fixes the minimum learning rate <code>alpha</code>. Use the settings provided in the cells below.
* Train a neural network with the same parameters as above but with cosine annealing learning rate. (That is, use <code>setup_model</code> and <code>model_compile</code> from above.)
* Report test accuracy and plot training loss, validation loss, training accuracy and validation accuracy.

In [ ]:
# found by hyperparameter tuning
best_initial_lr = 0.003
alpha = 0.01

### insert your code here ####



# Compare constant to cosine

One training run doesn't tell us much about any difference between constant learning rate and cosine annealing as the result could just be a statistical fluke. Let's repeat training several times and compare the means of training loss and accuracy.

## Task:
* write a loop that runs 5 times (you can run more often, but then it will take more time)
* in each iteration: draw a random train / test split via <code>get_train_test_split</code> above, train a model with constant learning rate, and one with cosine annealing, and record the training loss curve, the training accuracy curve and the test accuracy
* for constant learning rate and cosine annealing separately, compute the means over the loop iterations of training loss curve, training accuracy curve and test accuracy
* plot the means of training loss and training accuracy and report the means of test accuracy
* What do you observe?

In [ ]:
### insert your code here ###